In [7]:
import numpy as np
import pandas as pd
import time
import emoji
import datetime
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt1
import seaborn as sns
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('vader_lexicon')
start = time. time()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [8]:
def PreProcessing(filename):
    df = pd.read_csv("mabel.txt",header=None,on_bad_lines='skip',encoding='utf8')
    df=df.drop(0)
    print(df.head())
    df.columns=['Date','Text']
    return df

In [9]:
def SplitTextIntoColumns(df):
    Message = df["Text"].str.split("-",n=1,expand=True)
    print(Message.head())
    Message.columns=["Time","Chat"]
    print(Message.head())
    df["Time"]=Message["Time"]
    print(df.head())
    Message1=Message["Chat"].str.split(":",n=1,expand=True)
    print(Message1.head())
    df["Name"]=Message1[0]
    df["Text"]=Message1[1]
    print(df.head())
    df=df[["Date","Time","Name","Text"]]
    print(df.head())
    return df

In [10]:
def AnalyzeSentiment(df):
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    sid = SentimentIntensityAnalyzer()
    sid.polarity_scores(df['Text'][1])
    df['scores'] = df['Text'].apply(lambda comments: sid.polarity_scores(comments))
    print(df['scores'])
    df['compound'] = df['scores'].apply(lambda score_dict: score_dict['compound'])
    df['Negative'] = df['scores'].apply(lambda score_dict: score_dict['neg'])
    df['Positive'] = df['scores'].apply(lambda score_dict: score_dict['pos'])
    df['Neutral'] = df['scores'].apply(lambda score_dict: score_dict['neu'])
    print(df.head())
    df['comp_score'] = df['compound'].apply(lambda c : 'pos' if c>=0 else 'neg')
    print(df.head())
    posneg=pd.DataFrame(df['comp_score'].value_counts())
    print(f"Sentiment Analysis Results:\n{posneg}")
    return df

In [11]:
#Topic Modelling
def TopicModelling(df):
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.decomposition import LatentDirichletAllocation
    counter_vectorizer = CountVectorizer(stop_words='english',max_df=0.95, min_df=5, ngram_range=(1,2), max_features=5000)
    dtm = counter_vectorizer.fit_transform(df['Text']) # Converting Document Term Matrix
    lda = LatentDirichletAllocation(n_components=5, random_state=42)
    lda.fit(dtm)
    results=[]
    for index, topic in enumerate(lda.components_):
        print(f"Topic {index}:")
        result=([counter_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-10:]])
        results.append(result)
        print(result)
    topic_results = lda.transform(dtm)
    df['Topic'] = topic_results.argmax(axis=1)#Assign respective topics to each commentTent
    return results, df

In [12]:
df = PreProcessing("mabel.txt")

          0                                                  1
1  05/12/19   1:42 pm - Mabel Infoziant: Hi this is Mabel w...
2  05/12/19   1:42 pm - Mabel Infoziant: What’s your full name
3  05/12/19                      1:42 pm - AR❤: Ramisha Rani K
4  05/12/19                      1:42 pm - Mabel Infoziant: Ok
5  05/12/19   1:42 pm - Mabel Infoziant: ramisharanik@gmail...


In [14]:

df = SplitTextIntoColumns(df)
print(df.head())

           0                                                 1
1   1:42 pm    Mabel Infoziant: Hi this is Mabel we just spoke
2   1:42 pm             Mabel Infoziant: What’s your full name
3   1:42 pm                                AR❤: Ramisha Rani K
4   1:42 pm                                Mabel Infoziant: Ok
5   1:42 pm            Mabel Infoziant: ramisharanik@gmail.com
        Time                                              Chat
1   1:42 pm    Mabel Infoziant: Hi this is Mabel we just spoke
2   1:42 pm             Mabel Infoziant: What’s your full name
3   1:42 pm                                AR❤: Ramisha Rani K
4   1:42 pm                                Mabel Infoziant: Ok
5   1:42 pm            Mabel Infoziant: ramisharanik@gmail.com
       Date                                               Text       Time
1  05/12/19   1:42 pm - Mabel Infoziant: Hi this is Mabel w...   1:42 pm 
2  05/12/19   1:42 pm - Mabel Infoziant: What’s your full name   1:42 pm 
3  05/12/19           

In [15]:
df = AnalyzeSentiment(df)

1     {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
2     {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
3     {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
4     {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound...
5     {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
6     {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
7     {'neg': 0.0, 'neu': 0.27, 'pos': 0.73, 'compou...
8     {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
9     {'neg': 0.0, 'neu': 0.312, 'pos': 0.688, 'comp...
10    {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
11    {'neg': 0.0, 'neu': 0.915, 'pos': 0.085, 'comp...
12    {'neg': 0.0, 'neu': 0.761, 'pos': 0.239, 'comp...
13    {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
14    {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound...
15    {'neg': 0.0, 'neu': 0.896, 'pos': 0.104, 'comp...
16    {'neg': 0.0, 'neu': 0.609, 'pos': 0.391, 'comp...
17    {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
18    {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'comp

In [16]:
results, df =TopicModelling(df)

Topic 0:
['send', 'ok', 'mam']
Topic 1:
['send', 'mam', 'ok']
Topic 2:
['send', 'ok', 'mam']
Topic 3:
['ok', 'mam', 'send']
Topic 4:
['send', 'ok', 'mam']


In [17]:
print(results)

[['send', 'ok', 'mam'], ['send', 'mam', 'ok'], ['send', 'ok', 'mam'], ['ok', 'mam', 'send'], ['send', 'ok', 'mam']]


In [18]:
print(df.head())

       Date       Time              Name                             Text  \
1  05/12/19   1:42 pm    Mabel Infoziant   Hi this is Mabel we just spoke   
2  05/12/19   1:42 pm    Mabel Infoziant            What’s your full name   
3  05/12/19   1:42 pm                AR❤                   Ramisha Rani K   
4  05/12/19   1:42 pm    Mabel Infoziant                               Ok   
5  05/12/19   1:42 pm    Mabel Infoziant           ramisharanik@gmail.com   

                                              scores  compound  Negative  \
1  {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...     0.000       0.0   
2  {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...     0.000       0.0   
3  {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...     0.000       0.0   
4  {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound...     0.296       0.0   
5  {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...     0.000       0.0   

   Positive  Neutral comp_score  Topic  
1       0.0      1.0        pos      0 

In [19]:
print(df.iloc[0:10, :])

        Date       Time              Name  \
1   05/12/19   1:42 pm    Mabel Infoziant   
2   05/12/19   1:42 pm    Mabel Infoziant   
3   05/12/19   1:42 pm                AR❤   
4   05/12/19   1:42 pm    Mabel Infoziant   
5   05/12/19   1:42 pm    Mabel Infoziant   
6   05/12/19   1:43 pm    Mabel Infoziant   
7   05/12/19   1:43 pm                AR❤   
8   05/12/19   1:43 pm    Mabel Infoziant   
9   05/12/19   1:43 pm                AR❤   
10  05/12/19   1:43 pm    Mabel Infoziant   

                                               Text  \
1                    Hi this is Mabel we just spoke   
2                             What’s your full name   
3                                    Ramisha Rani K   
4                                                Ok   
5                            ramisharanik@gmail.com   
6                                    Your email Id?   
7                                           Yes Mam   
8    I will send 2 abstracts for u to start working   
9        

In [20]:
print(df.iloc[7])

Date                                                   05/12/19
Time                                                   1:43 pm 
Name                                            Mabel Infoziant
Text             I will send 2 abstracts for u to start working
scores        {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
compound                                                    0.0
Negative                                                    0.0
Positive                                                    0.0
Neutral                                                     1.0
comp_score                                                  pos
Topic                                                         3
Name: 8, dtype: object


In [24]:
print(results[4])

['send', 'ok', 'mam']
